# Xenium-Supervised Niche Labeling and H&E-Based Prediction for Interpretable Spatial Tissue Analysis

## Abstract
In this tutorial, you will learn how scientists can teach a computer to predict tissue niche types from H&E image-derived features. We use Xenium data only during training to provide biological labels, then train a model that can later predict niche types using H&E-derived features alone. The current notebook uses 12 processed samples: 2 samples for training and 10 samples for held-out validation/testing. You will run marker-gene analysis, GO enrichment analysis (GOEA), LLM-based label naming, and sample-level neural network classification in one reproducible pipeline.

## Core Idea
1. Use WSInsight outputs to get niche groups and H&E-derived features.
2. Use Xenium expression to understand the biology of each niche group.
3. Convert biological interpretation into niche labels.
4. Train a model on 2 samples and test it on the other 10 samples without Xenium at inference time.

## Problem Statement
Given per-cell H&E-derived niche features, learn a function $f_\theta(\cdot)$ that predicts niche subtype labels curated through Xenium-guided biological interpretation. At inference time, Xenium is not required.

**Introduction**

### Why we do this
Spatial transcriptomics can reveal rich biology, but it is expensive and not always available for every slide. H&E is much more scalable, so we want to transfer biological supervision from Xenium into a model that can later run on H&E-derived features alone.

### Scientific contribution
1. We use WSInsight graph-based niche discovery (`wsinsight niche`) to define niche groups from H&E context.
2. We assign biological meaning to each niche via marker genes + GOEA + LLM labeling.
3. We train and validate a supervised classifier that predicts niche subtype using only H&E-derived features.
4. We preserve a strict train/inference separation: Xenium is used to create labels, not for deployment-time prediction.
5. We evaluate on a strict sample-level split of 12 datasets (2 training samples, 10 held-out validation/testing samples).

**Methods Overview (as in a paper)**

### Data and pipeline entrypoints
- H&E slide files: `./data`
- H&E slide manifest: `./data/imagelist.txt`
- Xenium spatial manifest: `./data/sptxlist.tsv`
- Operational script: `./run-wsinsight-all.sh`

### Stage A. Upstream WSInsight processing
1. `wsinsight run`: H&E-based cell prediction
2. `wsinsight niche`: graph-based niche discovery (command name is niche)
3. `wsinsight import --include niche`: map Xenium expression to H&E cells

### Stage B. Label construction (training only)
1. Differential marker genes per niche group
2. GOEA for BP/CC/MF biological themes
3. LLM assigns one concise niche label per niche group

### Stage C. Sample-level supervised learning
1. Build niche-level summary vectors from H&E features for each sample
2. Train neural classifier on the 2 training samples only
3. Apply to the 10 held-out samples and report supervised metrics on the seen-label subset, with explicit coverage of excluded rows/samples

### Deployment principle
Use only H&E-derived features for niche prediction. Xenium is used for supervision, not for runtime inference.

## Stage 0: Execute WSInsight Pipeline

This stage produces the imported h5ad outputs used downstream.

Commands executed in order:
1. `wsinsight run`
2. `wsinsight niche`
3. `wsinsight import --include niche`

### Why this stage exists
WSInsight is the upstream step that turns H&E-derived image context into niche groups and niche features. The later marker-gene, GOEA, LLM labeling, and neural-network steps all depend on the outputs created here.

In plain terms, this stage does two jobs:
1. It turns local image context into a graph of related cells.
2. It turns that graph into niche groups, so each cell is assigned to a spatial community instead of being treated in isolation.

The `niche` step is the key part of that process. It combines graph construction, graph embedding, and clustering so the final niche groups are driven by neighborhood structure rather than by raw pixels alone.

### Method context for `wsinsight niche`
This command is the graph-learning and niche-discovery step. It does not just assign a label directly; it builds a graph over cells, learns an embedding, and then groups cells into niches.

At a high level, the workflow is:
1. Build a k-nearest-neighbor graph from cell features.
2. Learn a graph-aware representation with DGI.
3. Cluster the learned representation with Leiden or KMeans.
4. Save the resulting niche memberships and niche feature summaries back into the AnnData object.

#### kNN graph
- kNN means k-nearest neighbors.
- For each cell, WSInsight finds the `k` closest cells in feature space and connects them with edges.
- If $x_i$ is the feature vector for cell $i$, the method computes a distance such as Euclidean or cosine distance to other cells and keeps the `k` smallest distances.
- The graph adjacency matrix can be written as $A_{ij}=1$ if cell $j$ is among the `k` nearest neighbors of cell $i$, and $A_{ij}=0$ otherwise.
- In practice, this creates a sparse graph instead of a dense all-to-all matrix, which makes downstream learning faster and less noisy.
- Intuition: cells that look similar or share similar local context should be neighbors in the graph.

Why kNN is used:
- It converts local morphology into a neighborhood structure.
- It gives the model a way to say which cells belong together before any clustering is done.
- It reduces the problem from comparing every cell against every other cell to comparing each cell against only its nearest neighbors.

#### DGI (Deep Graph Infomax)
- DGI learns node embeddings by contrasting the real graph against a corrupted view of the graph.
- Suppose the graph encoder produces a node embedding $h_i$ for cell $i$ and a summary vector $s$ for the whole graph.
- DGI trains the encoder so the score $h_i^\top s$ is large for true node-summary pairs and small for fake pairs created by corrupting or shuffling the graph.
- A common objective is a binary cross-entropy style loss:
$$
\mathcal{L}_{DGI}=-\sum_{i\in V}\log\sigma(h_i^\top s)-\sum_{j\in\tilde V}\log\left(1-\sigma(\tilde h_j^\top s)\right)
$$
- Here, $\tilde V$ is the corrupted node set and $\sigma$ is the sigmoid function.
- The first term rewards true nodes that agree with the global summary, and the second term penalizes corrupted nodes that look too similar to that summary.
- In simple terms, DGI tries to learn a representation where a cell is close to its genuine neighborhood summary and far from a corrupted one.
- This gives WSInsight a learned graph representation before niche grouping.

Why DGI is used:
- Raw kNN edges only tell us which cells are neighbors; they do not yet give a compact representation of neighborhood pattern.
- DGI compresses the graph into embeddings that keep neighborhood information but remove some of the redundancy of the raw graph.
- The result is a feature space that is better suited for clustering than the original cell-level features alone.

#### Leiden
- Leiden is a community-detection algorithm.
- After the graph representation is built, Leiden partitions the graph into communities with high internal connectivity and low between-community leakage.
- The modularity-style idea is to compare the observed edge density inside a proposed community with the edge density expected under a random graph model.
- A classic modularity score is:
$$
Q=\frac{1}{2m}\sum_{i,j}\left(A_{ij}-\frac{k_i k_j}{2m}\right)\mathbf{1}(c_i=c_j)
$$
where $m$ is the number of edges, $k_i$ and $k_j$ are node degrees, and $\mathbf{1}(c_i=c_j)$ is 1 when cells $i$ and $j$ are in the same community.
- If the observed within-community connections are stronger than expected by chance, the score increases.
- In this notebook, those communities become the computational niche groups stored in `niche_*` columns.
- Intuition: if a set of cells is densely connected, Leiden tends to keep them together as one niche.

Why Leiden is used:
- It works directly on the graph, so the final niches respect the local neighborhood structure.
- It usually finds compact communities even when the graph has uneven density.
- It is a good fit when the biological question is about spatial neighborhoods rather than global Euclidean distance only.

#### KMeans
- KMeans is a centroid-based clustering method.
- It is an alternative way to partition learned embeddings into clusters when you want a simpler Euclidean clustering rule.
- Unlike Leiden, KMeans does not use graph community structure directly; it groups points by distance to cluster centers.
- The objective is to minimize the within-cluster sum of squares:
$$
\min_{\{\mu_k\}}\sum_{i=1}^{n}\min_{k\in\{1,\dots,K\}}\|x_i-\mu_k\|_2^2
$$
where $x_i$ is the embedding for cell $i$ and $\mu_k$ is the centroid of cluster $k$.
- The algorithm alternates between two steps: assign each point to the nearest centroid, then recompute the centroids as the mean of the assigned points.
- Intuition: each cell is assigned to the nearest centroid, and the centroids are updated iteratively until convergence.

Why KMeans is used:
- It gives a simple baseline clustering rule on the learned graph embeddings.
- It is easy to interpret because every niche is represented by a centroid.
- It can be useful when you want a fixed number of clusters or want to compare against Leiden.

### What to remember
- kNN builds the cell graph from local similarity.
- DGI learns graph-aware embeddings by separating real neighborhood structure from corrupted structure.
- Leiden turns the graph into niche communities by maximizing within-community connectivity.
- KMeans is a centroid-based alternative that minimizes within-cluster distance.

Conceptually, this stage transforms per-cell context into niche identities (`niche_*`) and niche feature columns (`niche_feature_normalized_*`).

The important idea is that `wsinsight niche` is not just clustering raw cells. It is doing graph learning first, then clustering the learned graph representation. That is why the output is more biologically meaningful than a plain pixel-based cluster assignment.

Dataset note for this notebook: Stage 0 produces imported outputs for all 12 samples; later Stage 3 trains on 2 samples and validates/tests on the other 10.

In [18]:
# Optional execution of the upstream WSInsight pipeline.
# Analogy: this is like deciding whether to actually start a washing machine or just read the instruction card first.
# This cell lets you choose whether to actually run the shell script or just print the commands.
import subprocess
from pathlib import Path

# Path object for the helper shell script in this project folder.
script_path = Path("run-wsinsight-all.sh")
if not script_path.exists():
    # Stop early with a clear message if the script file is missing.
    raise FileNotFoundError(f"Missing script: {script_path}")

# Safety switch: keep False for dry-run (show commands only).
# Set to True only when you are ready to run the full pipeline.
RUN_WSINSIGHT = False

if RUN_WSINSIGHT:
    # Actually execute: sh run-wsinsight-all.sh
    # check=True means Python raises an error if the command fails.
    subprocess.run(["sh", str(script_path)], check=True)
    print("WSInsight pipeline completed.")
else:
    # Dry-run mode: show exactly what would run, without changing files.
    print("RUN_WSINSIGHT=False; script not executed.")
    print(f"Planned command: sh {script_path}")
    print("The script runs:")
    print("  ./wsinsight-docker-run.sh . \"\" wsinsight run -i image-list:///workspace/data/imagelist.txt -o /workspace/outputs/ -m CellViT-SAM-H-x40 -b 12 -n 1")
    print("  ./wsinsight-docker-run.sh . \"\" wsinsight niche -i image-list:///workspace/data/imagelist.txt -o /workspace/outputs/ --niche-clusters 5")
    print("  ./wsinsight-docker-run.sh . \"\" wsinsight import -i image-list:///workspace/data/imagelist.txt -o /workspace/outputs/ -s sptx-list:///workspace/data/sptxlist.tsv --include niche")

RUN_WSINSIGHT=False; script not executed.
Planned command: sh run-wsinsight-all.sh
The script runs:
  ./wsinsight-docker-run.sh . "" wsinsight run -i image-list:///workspace/data/imagelist.txt -o /workspace/outputs/ -m CellViT-SAM-H-x40 -b 12 -n 1
  ./wsinsight-docker-run.sh . "" wsinsight niche -i image-list:///workspace/data/imagelist.txt -o /workspace/outputs/ --niche-clusters 5
  ./wsinsight-docker-run.sh . "" wsinsight import -i image-list:///workspace/data/imagelist.txt -o /workspace/outputs/ -s sptx-list:///workspace/data/sptxlist.tsv --include niche


**WSInsight Stage Output**
What you should see: either live execution logs (if `RUN_WSINSIGHT=True`) or a printed command sequence (default).

Generated outputs expected under `outputs/`:
- cell predictions from `wsinsight run`
- niche assignments/features from `wsinsight niche`
- imported Xenium-linked outputs from `wsinsight import --include niche`

Important carry-forward artifacts for this notebook:
- 12 Xenium-imported per-sample h5ad files in `outputs/imported-xenium/`
- `niche_*` niche membership columns
- `niche_feature_normalized_*` feature columns

Split note: these 12 samples are later split into 2 training samples and 10 held-out validation/testing samples.

**How To Run This Notebook (Step-by-Step)**
1. Run cells from top to bottom in order. Do not skip cells.
2. If a cell fails, fix that error first, then continue.
3. Keep the data paths unchanged unless your files are in a different location.
4. Do not run DE/GOEA cells before normalization/log1p is completed.
5. Save outputs at each stage; they are reused later in the pipeline.

### Expected outcome
By the end, you will train and evaluate a model that predicts niche labels from H&E-derived features on a 12-sample cohort with a strict split: 2 training samples and 10 held-out validation/testing samples.

### Install Dependencies
Installs required Python packages for the tutorial environment. If your environment already has these packages, you can skip this cell.

In [19]:
# Install required packages into the current notebook Python environment.
# Analogy: this is like stocking your kitchen with all ingredients before cooking.
# -q means quiet mode (less terminal output).
# You usually run this once per new environment.
# %pip install -q goatools mygene openai python-dotenv scanpy seaborn scikit-learn torch

### It's your turn. Leia ;-)
Read reference.ipynb if you stuck at somewhere. Good luck :-)

In [20]:
# Basic standard-library and data-science imports used throughout the notebook.
# Analogy: these are your toolbox items that will be reused in almost every step.
from pathlib import Path
import os
import numpy as np
import pandas as pd

# -----------------------------
# Global tutorial configuration
# -----------------------------
# Full fixed cohort of 12 samples for this notebook.
all_sample_stems = [
    "Human_Breast_Biomarkers_S1_Top_he_image.ome",
    "Human_Breast_Biomarkers_S1_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S1_Bot_he_image.ome",
    "Human_Breast_Biomarkers_S2_Top_he_image.ome",
    "Human_Breast_Biomarkers_S2_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S2_Bot_he_image.ome",
    "Human_Breast_Biomarkers_S3_Top_he_image.ome",
    "Human_Breast_Biomarkers_S3_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S3_Bot_he_image.ome",
    "Human_Breast_Biomarkers_S4_Top_he_image.ome",
    "Human_Breast_Biomarkers_S4_Mid_he_image.ome",
    "Human_Breast_Biomarkers_S4_Bot_he_image.ome",
]

# Fixed training subset (2 samples).
train_sample_stems = [
    "Human_Breast_Biomarkers_S1_Top_he_image.ome",
    "Human_Breast_Biomarkers_S1_Mid_he_image.ome",
]

# Held-out subset is "all minus train".
test_sample_stems = [s for s in all_sample_stems if s not in train_sample_stems]

# Folder containing per-sample imported .h5ad files from WSInsight import stage.
imported_xenium_dir = Path("outputs/v3/imported-xenium")

# Central configuration dictionary so later cells can reuse the same settings.
CFG = {
    "slides_dir": Path("data"),
    "image_list_txt": Path("data/imagelist.txt"),
    "sptx_map_tsv": Path("data/sptxlist.tsv"),
    "run_script": Path("run-wsinsight-all.sh"),
    "imported_xenium_dir": imported_xenium_dir,
    "sample_split_dir": Path("outputs/v3/nn_sample_split"),
    "all_sample_stems": all_sample_stems,
    "train_sample_stems": train_sample_stems,
    "test_sample_stems": test_sample_stems,
    "train_sample_count": len(train_sample_stems),
    "test_sample_count": len(test_sample_stems),
    "outdir": Path("results/goea_multi_niche"),
    "seed": 42,
    "cells_per_draw": 100,
    "nn_epochs": 40,
    "nn_batch_size": 128,
    "nn_lr": 1e-3,
    "nn_weight_decay": 1e-4,
}

# Hard guardrail: this tutorial assumes exactly a 12/2/10 split.
if CFG["train_sample_count"] != 2 or CFG["test_sample_count"] != 10 or len(CFG["all_sample_stems"]) != 12:
    raise RuntimeError("Sample split must be exactly: 12 total, 2 training, 10 held-out test.")

# Set NumPy random seed for reproducible random operations.
np.random.seed(CFG["seed"])
# Create output directories now so later save operations do not fail.
CFG["outdir"].mkdir(parents=True, exist_ok=True)
CFG["sample_split_dir"].mkdir(parents=True, exist_ok=True)

# Print a readable summary so users can verify configuration quickly.
print("Configuration loaded.")
print("All samples:", len(CFG["all_sample_stems"]))
print("Training samples:", CFG["train_sample_stems"])
print("Held-out test samples:", CFG["test_sample_stems"])
print(pd.Series({k: str(v) for k, v in CFG.items() if "dir" in k or "txt" in k or "tsv" in k or "script" in k}).to_string())

Configuration loaded.
All samples: 12
Training samples: ['Human_Breast_Biomarkers_S1_Top_he_image.ome', 'Human_Breast_Biomarkers_S1_Mid_he_image.ome']
Held-out test samples: ['Human_Breast_Biomarkers_S1_Bot_he_image.ome', 'Human_Breast_Biomarkers_S2_Top_he_image.ome', 'Human_Breast_Biomarkers_S2_Mid_he_image.ome', 'Human_Breast_Biomarkers_S2_Bot_he_image.ome', 'Human_Breast_Biomarkers_S3_Top_he_image.ome', 'Human_Breast_Biomarkers_S3_Mid_he_image.ome', 'Human_Breast_Biomarkers_S3_Bot_he_image.ome', 'Human_Breast_Biomarkers_S4_Top_he_image.ome', 'Human_Breast_Biomarkers_S4_Mid_he_image.ome', 'Human_Breast_Biomarkers_S4_Bot_he_image.ome']
slides_dir                                   data
image_list_txt                 data/imagelist.txt
sptx_map_tsv                    data/sptxlist.tsv
run_script                   run-wsinsight-all.sh
imported_xenium_dir    outputs/v3/imported-xenium
sample_split_dir       outputs/v3/nn_sample_split
outdir                   results/goea_multi_niche


In [21]:
# Validate key input paths and verify all configured imported .h5ad files are present.
# Analogy: checking all files are present is like confirming every passenger is on the bus before departure.
required_paths = [
    CFG["slides_dir"],
    CFG["image_list_txt"],
    CFG["sptx_map_tsv"],
    CFG["imported_xenium_dir"],
]

# Loop through important paths and fail early if anything is missing.
for p in required_paths:
    if not p.exists():
        raise FileNotFoundError(f"Missing required path: {p}")

# Read mapping table: each row links Xenium path to sample_id.
map_df = pd.read_csv(CFG["sptx_map_tsv"], sep="\t", header=None, names=["sptx_path", "sample_id"])
if map_df.empty:
    raise RuntimeError("data/sptxlist.tsv is empty.")

# Confirm that every configured sample has a corresponding imported .h5ad file.
expected_h5ad_paths = [CFG["imported_xenium_dir"] / f"{stem}.h5ad" for stem in CFG["all_sample_stems"]]
missing_h5ad = [str(p) for p in expected_h5ad_paths if not p.exists()]
if missing_h5ad:
    raise FileNotFoundError(f"Missing imported h5ad files for configured samples: {missing_h5ad}")

# Print quick sanity-check outputs for users.
print("All required inputs found.")
print("\nSample mapping preview:")
print(map_df.head().to_string(index=False))
print(f"\nTotal mapping rows: {len(map_df)}")
print(f"Imported h5ad files verified: {len(expected_h5ad_paths)}")

All required inputs found.

Sample mapping preview:
                                          sptx_path                                   sample_id
/workspace/data/Human_Breast_Biomarkers_S1_Top_outs Human_Breast_Biomarkers_S1_Top_he_image.ome
/workspace/data/Human_Breast_Biomarkers_S1_Bot_outs Human_Breast_Biomarkers_S1_Bot_he_image.ome
/workspace/data/Human_Breast_Biomarkers_S2_Top_outs Human_Breast_Biomarkers_S2_Top_he_image.ome
/workspace/data/Human_Breast_Biomarkers_S2_Mid_outs Human_Breast_Biomarkers_S2_Mid_he_image.ome
/workspace/data/Human_Breast_Biomarkers_S2_Bot_outs Human_Breast_Biomarkers_S2_Bot_he_image.ome

Total mapping rows: 12
Imported h5ad files verified: 12


In [22]:
# Import libraries needed for plotting, Scanpy processing, and GO enrichment analysis.
# Analogy: this is laying out all lab instruments on the bench before the experiment starts.
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import mygene

from goatools.base import download_go_basic_obo, download_ncbi_associations
from goatools.obo_parser import GODag
from goatools.anno.genetogo_reader import Gene2GoReader
from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS

# Make Scanpy output less noisy and figures readable in notebook.
sc.settings.verbosity = 1
sc.set_figure_params(dpi=110, frameon=False)

# MyGene client used later to map gene symbols to Entrez IDs for GOEA.
mg = mygene.MyGeneInfo()
print("Imports ready.")

Imports ready.


In [23]:
# Load and concatenate all 12 imported AnnData objects into one cohort object.
# Analogy: this is like combining 12 separate spreadsheets into one master workbook.
# This unified object is used for cohort-level interpretation steps.
expected_h5ad_paths = [CFG["imported_xenium_dir"] / f"{stem}.h5ad" for stem in CFG["all_sample_stems"]]
cohort_adatas = []

# Read each sample file and attach sample_id so we can track origin after concatenation.
for stem, path in zip(CFG["all_sample_stems"], expected_h5ad_paths):
    print(f"Loading {stem}: {path}")
    ad = sc.read_h5ad(path)
    ad.obs = ad.obs.copy()
    ad.obs["sample_id"] = stem
    cohort_adatas.append(ad)

# Concatenate samples into one AnnData.
# join="inner" keeps only shared variables across all samples.
adata = sc.concat(
    cohort_adatas,
    join="inner",
    merge="same",
    label="cohort_key",
    keys=CFG["all_sample_stems"],
    index_unique="__",
)

# Keep a working copy to safely modify in downstream steps.
adata_work = adata.copy()

# Sanity check: confirm all configured samples are present.
n_samples_loaded = int(adata_work.obs["sample_id"].nunique())
if n_samples_loaded != len(CFG["all_sample_stems"]):
    raise RuntimeError(
        f"Cohort concat mismatch: expected {len(CFG['all_sample_stems'])} samples, got {n_samples_loaded}"
    )

# Print useful summary information for quick inspection.
print("Cohort AnnData loaded.")
print("Samples in cohort:", n_samples_loaded)
print(adata)
print("obs columns:", len(adata_work.obs.columns))
print("var genes:", adata_work.n_vars)
print("cells:", adata_work.n_obs)
print(adata_work.obs["sample_id"].value_counts().sort_index().to_string())

Loading Human_Breast_Biomarkers_S1_Top_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S1_Top_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S1_Mid_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S1_Mid_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S1_Bot_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S1_Bot_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S2_Top_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S2_Top_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S2_Mid_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S2_Mid_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S2_Bot_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S2_Bot_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S3_Top_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biomarkers_S3_Top_he_image.ome.h5ad
Loading Human_Breast_Biomarkers_S3_Mid_he_image.ome: outputs/v3/imported-xenium/Human_Breast_Biom

In [24]:
if "sample_id" not in adata_work.obs.columns:
    print("Warning: sample_id metadata missing. Creating a fallback sample_id='unknown'.")
    adata_work.obs = adata_work.obs.copy()
    adata_work.obs["sample_id"] = "unknown"

if adata_work.obs["sample_id"].nunique() != len(CFG["all_sample_stems"]):
    print(
        "Warning: preprocessing expected all configured samples, but counts differ. "
        "Continuing for educational flow."
    )

sc.pp.calculate_qc_metrics(adata_work, percent_top=None, log1p=False, inplace=True)

if "counts" not in adata_work.layers:
    adata_work.layers["counts"] = adata_work.X.copy()

# Avoid accidental double-normalization by checking if log1p metadata already exists.
if "log1p" not in adata_work.uns:
    sc.pp.normalize_total(adata_work, target_sum=1e4)
    sc.pp.log1p(adata_work)
    print("Applied normalize_total + log1p.")
else:
    print("Data already appears log-normalized; skipped re-normalization.")

print("Cells:", adata_work.n_obs, "Genes:", adata_work.n_vars)




/home/leiah/.conda/envs/spie2027/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Applied normalize_total + log1p.
Cells: 3442976 Genes: 280


In [25]:
# Detect niche membership columns and normalized feature columns automatically.
# Analogy: this is like scanning a form to find the exact columns you need before calculations.
niche_list = sorted(
    [c for c in adata_work.obs.columns if c.startswith("niche_") and c[6:].isnumeric()],
    key=lambda x: int(x.split("_")[1]),
)
feature_cols = sorted([c for c in adata_work.obs.columns if c.startswith("niche_feature_normalized_")])

if not niche_list:
    raise RuntimeError("No niche_* membership columns found.")
if not feature_cols:
    raise RuntimeError("No niche_feature_normalized_* feature columns found.")

# Print summary to confirm expected columns are present.
print(f"Detected {len(niche_list)} niche groups")
print(f"Detected {len(feature_cols)} normalized niche features")
print("First 10 niche-group columns:", niche_list[:10])
print("Feature columns:", feature_cols)

Detected 10 niche groups
Detected 18 normalized niche features
First 10 niche-group columns: ['niche_0', 'niche_1', 'niche_2', 'niche_3', 'niche_4', 'niche_5', 'niche_6', 'niche_7', 'niche_8', 'niche_9']
Feature columns: ['niche_feature_normalized_k0_background', 'niche_feature_normalized_k0_connective', 'niche_feature_normalized_k0_dead', 'niche_feature_normalized_k0_epithelial', 'niche_feature_normalized_k0_inflammatory', 'niche_feature_normalized_k0_neoplastic', 'niche_feature_normalized_k1_background', 'niche_feature_normalized_k1_connective', 'niche_feature_normalized_k1_dead', 'niche_feature_normalized_k1_epithelial', 'niche_feature_normalized_k1_inflammatory', 'niche_feature_normalized_k1_neoplastic', 'niche_feature_normalized_k2_background', 'niche_feature_normalized_k2_connective', 'niche_feature_normalized_k2_dead', 'niche_feature_normalized_k2_epithelial', 'niche_feature_normalized_k2_inflammatory', 'niche_feature_normalized_k2_neoplastic']


In [28]:
# Prepare the train-only cohort for marker-gene testing.
# Analogy: this is like choosing only the training slides before measuring which genes stand out.
from tqdm.auto import tqdm

# Educational mode fallback for missing sample_id.
if "sample_id" not in adata_work.obs.columns:
    print(
        "Warning: adata_work is missing sample_id. "
        "Using the full cohort as label-construction data for this demo run."
    )
    adata_label_work = adata_work.copy()
    label_train_stems = set()
else:
    # Build train-only AnnData for label construction (strict split discipline).
    label_train_stems = set(CFG["train_sample_stems"])
    label_train_mask = adata_work.obs["sample_id"].isin(label_train_stems)
    adata_label_work = adata_work[label_train_mask].copy()

# Verify expected number of train samples in the label-construction cohort.
n_label_samples = int(adata_label_work.obs["sample_id"].nunique()) if "sample_id" in adata_label_work.obs.columns else 0
if label_train_stems and n_label_samples != len(label_train_stems):
    print(
        "Warning: label-construction cohort mismatch. "
        f"Expected {len(label_train_stems)} train samples, got {n_label_samples}. "
        "Continuing for educational flow."
    )

# Minimum number of positive/negative cells required for stable DE testing.
min_positive_cells = 20
all_marker_tables = []
marker_summary = []

In [31]:
# Differential expression (DE) across all detected niche groups.
#
# Learning goal for beginners:
# For each niche group, compare gene expression in two sets of cells:
#   - in-group cells: membership == 1
#   - out-group cells: membership != 1
# If a gene is much higher in the in-group and statistically significant,
# that gene is a candidate marker for the niche biology.

# Cache paths so repeated runs can skip expensive DE recomputation.
markers_all_path = CFG["outdir"] / "markers_all_niche.csv"
markers_filt_path = CFG["outdir"] / "markers_filt_niche.csv"
marker_summary_path = CFG["outdir"] / "marker_summary_by_niche.csv"
marker_cache_meta_path = CFG["outdir"] / "marker_de_cache_meta.csv"

# cache_meta_now = {
#     "train_samples": "|".join(sorted(label_train_stems)),
#     "niche_groups": "|".join(niche_list),
#     "min_positive_cells": int(min_positive_cells),
#     "label_cells": int(adata_label_work.n_obs),
# }

# use_cache = False
# if markers_all_path.exists() and markers_filt_path.exists() and marker_summary_path.exists() and marker_cache_meta_path.exists():
#     try:
#         cache_meta_old = pd.read_csv(marker_cache_meta_path)
#         if not cache_meta_old.empty:
#             row = cache_meta_old.iloc[0].to_dict()
#             use_cache = (
#                 str(row.get("train_samples", "")) == cache_meta_now["train_samples"]
#                 and str(row.get("niche_groups", "")) == cache_meta_now["niche_groups"]
#                 and int(row.get("min_positive_cells", -1)) == cache_meta_now["min_positive_cells"]
#                 and int(row.get("label_cells", -1)) == cache_meta_now["label_cells"]
#             )
#     except Exception:
#         use_cache = False

# if use_cache:
#     markers_all = pd.read_csv(markers_all_path)
#     markers_filt_all = pd.read_csv(markers_filt_path)
#     marker_summary_df = pd.read_csv(marker_summary_path)
#     print("Loaded cached marker DE tables from disk.")
# else:
# Reset containers each run so repeated execution does not duplicate results.
all_marker_tables = []
marker_summary = []

# We loop over every detected niche, so this is never hardcoded to 5 or 10.
for niche_col in tqdm(niche_list, desc="Marker DE by niche group", unit="group"):
    # Build boolean masks for in-group and out-group.
    pos_mask = adata_label_work.obs[niche_col].fillna(0).astype(int).eq(1)
    n_pos = int(pos_mask.sum())
    n_neg = int((~pos_mask).sum())

    # We skip tiny groups because statistical tests become unstable
    # when one side has too few cells.
    if n_pos < min_positive_cells or n_neg < min_positive_cells:
        marker_summary.append({"niche_group": niche_col, "n_pos": n_pos, "n_neg": n_neg, "status": "skipped_small_group"})
        continue

    # group_col is a temporary label used by Scanpy to define two classes.
    group_col = f"{niche_col}_group"
    key_added = f"de_{niche_col}"
    adata_label_work.obs[group_col] = pd.Categorical(np.where(pos_mask, "in", "out"), categories=["in", "out"])

    # Wilcoxon rank-sum compares expression rank distributions between groups.
    # It is non-parametric: it does not assume expression follows a normal curve.
    sc.tl.rank_genes_groups(
        adata_label_work,
        groupby=group_col,
        groups=["in"],
        reference="out",
        method="wilcoxon",
        pts=True,
        key_added=key_added,
    )

    # Convert DE output into a table and tag which niche produced it.
    mk = sc.get.rank_genes_groups_df(adata_label_work, group="in", key=key_added)
    mk["niche_group"] = niche_col
    mk["n_pos"] = n_pos
    mk["n_neg"] = n_neg
    all_marker_tables.append(mk)
    marker_summary.append({"niche_group": niche_col, "n_pos": n_pos, "n_neg": n_neg, "status": "ok"})

# Combine DE tables from all niches.
markers_all = pd.concat(all_marker_tables, ignore_index=True) if all_marker_tables else pd.DataFrame()

# Keep genes that pass both:
#   1) adjusted p-value < 0.05 (statistical significance after correction)
#   2) log fold change > 0.25 (practical effect size threshold)
markers_filt_all = (
    markers_all.query("pvals_adj < 0.05 and logfoldchanges > 0.25").copy()
    if not markers_all.empty else pd.DataFrame()
)
marker_summary_df = pd.DataFrame(marker_summary)

# Save reproducible outputs for downstream GOEA and label construction.
markers_all.to_csv(markers_all_path, index=False)
markers_filt_all.to_csv(markers_filt_path, index=False)
marker_summary_df.to_csv(marker_summary_path, index=False)
# pd.DataFrame([cache_meta_now]).to_csv(marker_cache_meta_path, index=False)
# print("Computed marker DE and refreshed cache on disk.")

# QC output: print complete per-niche summary (not just head).
n_niches_detected = len(niche_list)
n_niches_tested_ok = int((marker_summary_df["status"] == "ok").sum()) if not marker_summary_df.empty and "status" in marker_summary_df.columns else 0
n_niches_skipped = int((marker_summary_df["status"] != "ok").sum()) if not marker_summary_df.empty and "status" in marker_summary_df.columns else 0

print("Label-construction samples:", sorted(label_train_stems))
print(f"Niche groups detected: {n_niches_detected}")
print(f"Niche groups tested successfully: {n_niches_tested_ok}")
print(f"Niche groups skipped (too few cells): {n_niches_skipped}")
print("\nPer-niche DE summary:")
print(marker_summary_df.to_string(index=False))
print("\nTotal marker rows:", len(markers_all))
print("Filtered marker rows:", len(markers_filt_all))
print("Samples represented for label construction:", adata_label_work.obs["sample_id"].nunique())
print("Marker DE cache metadata:", marker_cache_meta_path)

Marker DE by niche group:   0%|          | 0/10 [00:00<?, ?group/s]

Label-construction samples: ['Human_Breast_Biomarkers_S1_Mid_he_image.ome', 'Human_Breast_Biomarkers_S1_Top_he_image.ome']
Niche groups detected: 10
Niche groups tested successfully: 10
Niche groups skipped (too few cells): 0

Per-niche DE summary:
niche_group  n_pos  n_neg status
    niche_0 133740 451256     ok
    niche_1  33300 551696     ok
    niche_2  61155 523841     ok
    niche_3  19278 565718     ok
    niche_4  38358 546638     ok
    niche_5 163535 421461     ok
    niche_6   6486 578510     ok
    niche_7   7572 577424     ok
    niche_8  74241 510755     ok
    niche_9  37871 547125     ok

Total marker rows: 2800
Filtered marker rows: 794
Samples represented for label construction: 2
Marker DE cache metadata: results/goea_multi_niche/marker_de_cache_meta.csv


In [32]:
# GOEA setup utilities using the training-only label-construction cohort.
# Analogy: this prepares the dictionary and rulebook before enrichment scoring.
def extract_symbol_to_entrez(df):
    """Convert MyGene results to {GENE_SYMBOL: ENTREZ_ID} dictionary."""
    if isinstance(df, pd.DataFrame):
        x = df.reset_index().rename(columns={"query": "query_symbol"})
    else:
        x = pd.DataFrame(df)
    if "query_symbol" not in x.columns and "query" in x.columns:
        x = x.rename(columns={"query": "query_symbol"})

    # Keep only rows with valid Entrez IDs, normalize symbol format, and cast type.
    x = x.dropna(subset=["entrezgene"]).copy()
    x["query_symbol"] = x["query_symbol"].astype(str).str.upper()
    x["entrezgene"] = x["entrezgene"].astype(int)
    return dict(zip(x["query_symbol"], x["entrezgene"]))

if "adata_label_work" not in globals():
    raise RuntimeError("adata_label_work missing. Run marker-discovery cell first.")

# Build GOEA population (gene universe) from training-only AnnData variable names.
universe_symbols = pd.Index(adata_label_work.var_names).astype(str).str.upper().unique().tolist()
uni_q = mg.querymany(
    universe_symbols, scopes="symbol", fields="entrezgene,symbol",
    species="human", as_dataframe=True, returnall=False, verbose=False
)
uni_map = extract_symbol_to_entrez(uni_q)
population_ids = set(uni_map.values())

# Download/load GO ontology and gene-to-GO mappings.
obo_path = download_go_basic_obo(str(CFG["outdir"] / "go-basic.obo"))
gene2go_path = download_ncbi_associations(str(CFG["outdir"] / "gene2go"))
go_dag = GODag(obo_path)
objanno = Gene2GoReader(gene2go_path, taxids=[9606])
ns2assoc = objanno.get_ns2assc()

print("Mapped population IDs:", len(population_ids))

$ get http://purl.obolibrary.org/obo/go/go-basic.obo
requests.get(http://purl.obolibrary.org/obo/go/go-basic.obo, stream=True)
  WROTE: /workspace/leiah_Insync/leiahuang3352@gmail.com/Google Drive/SPIE2027/results/goea_multi_niche/.go-basic.obo.tmp.14cvbt94

$ get ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/gene2go.gz
FTP RETR ftp.ncbi.nlm.nih.gov gene/DATA gene2go.gz -> /workspace/leiah_Insync/leiahuang3352@gmail.com/Google Drive/SPIE2027/results/goea_multi_niche/.gene2go.gz.tmp.ezsro4_i
$ gunzip results/goea_multi_niche/gene2go.gz
results/goea_multi_niche/go-basic.obo: fmt(1.2) rel(2026-06-15) 41,535 Terms
HMS:0:02:31.384163 445,874 annotations, 20,362 genes, 17,984 GOs, 1 taxids READ: results/goea_multi_niche/gene2go 
Mapped population IDs: 277
